# Batched Quotes & OHLCV Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apextick/apextick-starter-kit/blob/main/notebooks/01_batched_quotes_and_ohlcv_quickstart.ipynb)

Replace one-request-per-ticker loops with **a single request for 50 symbols**, then pull **daily OHLCV bars** and **pre-computed technical indicators** (RSI-14, MACD, SMA-200) with the official `apextick` Python client.

**Plan & quota:** sections 1–2 run on the free Hobbyist plan (500 requests/day) and use 2 requests. Section 3 needs Trader Pro or above and uses 3 more; on the free plan it explains the upgrade instead of failing.

In [ ]:
# The official client uses only the standard library; pandas and matplotlib come with Colab.
%pip install -q "git+https://github.com/apextick/apextick-starter-kit.git#subdirectory=python"

## Add your API key

In Colab, open **Secrets** (the key icon in the left sidebar), add `APEXTICK_API_KEY` with your key and allow this notebook to use it. Anywhere else, set the `APEXTICK_API_KEY` environment variable, or paste the key when prompted. No key yet? Get a free one at [dash.apextick.com/register](https://dash.apextick.com/register).

In [ ]:
import os
from getpass import getpass

try:  # Colab Secrets
    from google.colab import userdata
    os.environ.setdefault("APEXTICK_API_KEY", userdata.get("APEXTICK_API_KEY") or "")
except Exception:
    pass
if not os.environ.get("APEXTICK_API_KEY"):
    os.environ["APEXTICK_API_KEY"] = getpass("ApexTick API key (dk_...): ")

import pandas as pd
import matplotlib.pyplot as plt
from apextick import ApexTickAPIError, ApexTickClient, to_dataframe

client = ApexTickClient()  # reads APEXTICK_API_KEY
client

## 1. One request instead of fifty

The usual scraper pattern makes one HTTP call per ticker:

```python
for symbol in watchlist:          # 50 symbols -> 50 requests and 50 round trips
    price = fetch_quote(symbol)
```

`/v1/market/quote` takes up to 50 comma-separated symbols, so a whole watchlist of stocks, ETFs, crypto and FX is **one request** and one unit of your daily quota. (`client.quote()` splits longer lists into batches of 50 for you.)

In [ ]:
import time

WATCHLIST = [
    # US large caps
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "BRK-B", "JPM", "V",
    "UNH", "XOM", "JNJ", "WMT", "MA", "PG", "AVGO", "HD", "CVX", "LLY",
    "MRK", "PEP", "KO", "COST", "ABBV", "ADBE", "CRM", "NFLX", "AMD", "ORCL",
    "INTC", "DIS", "BA", "NKE",
    # ETFs
    "SPY", "QQQ", "IWM", "DIA", "TLT", "GLD", "XLK", "XLF", "XLE", "ARKK",
    # Crypto and FX
    "BTC-USD", "ETH-USD", "SOL-USD", "EURUSD=X", "GBPUSD=X", "USDJPY=X",
]
assert len(WATCHLIST) == 50

started = time.perf_counter()
quotes = client.quote(WATCHLIST)
print(f"{len(quotes)} quotes in 1 request, {(time.perf_counter() - started) * 1000:.0f} ms round trip")

board = pd.DataFrame(quotes).set_index("symbol")
board["pct_from_52w_high"] = (board["price"] / board.reindex(columns=["high_52w"])["high_52w"] - 1) * 100
columns = ["name", "price", "change_percent", "volume", "high_52w", "low_52w", "pct_from_52w_high"]
board.reindex(columns=columns).sort_values("change_percent", ascending=False).head(10)

In [ ]:
# Session leaders and laggards across the whole board
ranked = board["change_percent"].sort_values()
pd.concat([ranked.head(5), ranked.tail(5)]).plot.barh(figsize=(8, 4), title="Session % change: 5 laggards and 5 leaders");

## 2. Historical OHLCV bars

`client.candles()` returns bars oldest first; `limit` keeps the most recent ones. Resolutions: `1m`, `5m`, `15m`, `30m`, `1h`, `1d`, `1wk`, `1mo`. History depth depends on your plan (1 year of daily bars on Hobbyist).

In [ ]:
SYMBOL = "AAPL"
bars = client.candles(SYMBOL, resolution="1d", limit=180)

ohlcv = pd.DataFrame(bars)
ohlcv["time"] = pd.to_datetime(ohlcv["time"])
ohlcv = ohlcv.set_index("time").sort_index()
print(f"{len(ohlcv)} daily bars for {SYMBOL}: {ohlcv.index[0].date()} to {ohlcv.index[-1].date()}")
ohlcv.tail()

In [ ]:
ax = ohlcv["close"].plot(figsize=(10, 4), title=f"{SYMBOL} daily close", label="close")
ohlcv["close"].rolling(20).mean().plot(ax=ax, label="20-day mean")
ohlcv["close"].rolling(50).mean().plot(ax=ax, label="50-day mean")
ax.legend();

## 3. Pre-computed technical indicators (Trader Pro+)

Instead of downloading a year of bars to compute RSI, MACD or a 200-day moving average yourself, ask for the values directly from `/v1/analytics/indicators`. Available: `sma20`, `sma50`, `sma200`, `rsi14`, `macd14`, `atr14`, `stoch_k14`, `volatility30`.

On the free plan the API answers HTTP 403; the cell below explains the upgrade instead of failing.

In [ ]:
rows = []
for symbol in ["AAPL", "NVDA", "SPY"]:
    try:
        values = client.technicals(symbol, indicators=["rsi14", "macd14", "sma200"])["indicators"]
    except ApexTickAPIError as err:
        if err.status_code == 403:
            print(f"Technical indicators need Trader Pro or above ({err.message}).")
            print("Upgrade at https://dash.apextick.com/billing/pricing")
            break
        raise
    rows.append({"symbol": symbol, "price": board["price"].get(symbol), **values})

technicals = pd.DataFrame(rows).set_index("symbol") if rows else pd.DataFrame()
if not technicals.empty:
    technicals["pct_vs_sma200"] = (technicals["price"] / technicals["sma200"] - 1) * 100
    technicals["rsi_zone"] = pd.cut(technicals["rsi14"], [0, 30, 70, 100], labels=["oversold", "neutral", "overbought"])
technicals

## Next steps

- **Options:** [`02_historical_options_greeks_backtest.ipynb`](https://colab.research.google.com/github/apextick/apextick-starter-kit/blob/main/notebooks/02_historical_options_greeks_backtest.ipynb) pulls live option chains with Black-Scholes Greeks and the daily history of a single contract.
- **Docs:** every endpoint and field at [docs.apextick.com](https://docs.apextick.com).
- **AI agents:** the same data is available to Claude, Cursor and other MCP clients: [docs.apextick.com/mcp](https://docs.apextick.com/mcp).